# Controlling Dobot

In [ ]:
from pathlib import Path
from dobot1_control import DobotControl
import numpy as np

# fix permissions possibly:
#   sudo chmod o+rw /dev/ttyACM0
# or for many places:
#   for dev in /dev/ttyACM*; do echo $dev; sudo chmod o+rw "$dev"; done
comport = str(next(Path("/dev").glob("ttyACM*")))
dobot = DobotControl(comport, endEffectorOffset=(49., 64.), accelOffset=(1016.39, 1009.08), accelConversion=509.00, plot=True, debug=True)
# percentage of maximum joint speed
speed = 0.5
# percentage of maximum joint acceleration
acceleration = 0.5
curpos = dobot.pos

In [ ]:
dobot.pos

In [ ]:
# this should be a no-op that should not fail
dobot.MoveWithSpeed((curpos+(0,0,0),), speed, acceleration)

In [ ]:
# do a rectangular motion for testing the motion planning which must stop at the edge
#nextPos += (10,0,0)  # adjust for relative movement
dobot.MoveWithSpeed((curpos, curpos + (-2, 0, 0), curpos + (20, 0, 0), curpos + (20, 20, 0), curpos + (20, 22, 0), curpos), speed, acceleration)

In [ ]:
pts = [
    np.array([230.94713508,   0.        ,  94.01584856]),
    np.array([228.98939573,  12.36067977,  94.01584856]),
    np.array([223.30781486,  23.51141009,  94.01584856]),
    np.array([214.45854517,  32.36067977,  94.01584856]),
    np.array([203.30781486,  38.04226065,  94.01584856]),
    np.array([190.94713508,  40.        ,  94.01584856]),
    np.array([178.58645531,  38.04226065,  94.01584856]),
    np.array([167.43572499,  32.36067977,  94.01584856]),
    np.array([158.58645531,  23.51141009,  94.01584856]),
    np.array([152.90487443,  12.36067977,  94.01584856]),
    np.array([1.50947135e+02, 4.89858720e-15, 9.40158486e+01]),
    np.array([152.90487443, -12.36067977,  94.01584856]),
]
dobot.MoveWithSpeed(pts, speed, acceleration)

In [ ]:
dobot.MoveWithSpeed((curpos + (40, 40, 0), curpos + (40, 45, 0), curpos + (40, -40, 0)), speed, acceleration)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def circle_points(center, radius, n=100):
    cx, cy = center[:2]
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    x = cx + radius * np.cos(theta)
    y = cy + radius * np.sin(theta)
    pts = np.column_stack((x, y))
    # combine and repeat the first point at the end
    return np.vstack((pts, pts[0]))

def plot_points(x, y, center=None, ax=None, label=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(x, y, marker='o', markersize=4, linestyle='-', label=label)
    if center is not None:
        ax.scatter(*center, color='red')
    ax.set_aspect('equal', 'box')
    ax.grid(True)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    if label is not None:
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    return ax

In [ ]:
# first point
first = (220, 0., 15)
dobot.MoveWithSpeed((first,), speed, acceleration)
curpos = dobot.pos
curpos

In [ ]:
radius = 40
pts = circle_points(curpos, radius, n=20)
plot_points(pts[:,0], pts[:,1], curpos)
zPos = 11
pts3 = [np.array((*pt, zPos)) for pt in pts]
pts3

In [ ]:
dobot.MoveWithSpeed(pts3, .5*speed, acceleration)
# back to start pos (center)
dobot.MoveWithSpeed(curpos, .5*speed, acceleration)

In [ ]:
for pt in pts3[:3]:
    print(f"Going to {pt} ..")
    dobot.MoveWithSpeed(pt, speed, acceleration)

In [ ]:
from pathlib import Path
from dobot1_control import svg_to_polylines
fn = Path("../testdata/rect-circle-top.svg")
paths, bounds = svg_to_polylines(fn, upper_left=(50, 120))
bounds, paths

In [ ]:
ax = None
for i, path in enumerate(paths):
    ax = plot_points(path[:,0], path[:,1], ax=ax, label=f"Path {i}")